# Phase 3 — Analysis & Visualisation

**Goal:** Load results from `02_main_experiments.ipynb`, produce all figures
for the report, and save the final report table as a CSV.

**Expects:**
- `results/results_main.csv` (from Phase 2)
- `checkpoints/` with all three SSL encoder checkpoints

**Outputs:**
- `figures/` folder with all plots
- `results/report_table.csv` — mean ± std table for the report
- Inline matplotlib figures (displayed in notebook)

## 1 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!git clone https://github.com/Friedrich-233/ST456_GroupProject.git /content/ST456_GroupProject
!pip install timm seaborn scikit-learn -q
import sys
sys.path.insert(0, '/content/ST456_GroupProject')

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RESULTS_DIR   = Path('/content/results')
CHECKPOINT_DIR = Path('/content/checkpoints')
FIGURE_DIR    = Path('/content/figures')
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

results_df = pd.read_csv(RESULTS_DIR / 'results_main.csv')
print(f"Loaded {len(results_df)} experiment rows")
display(results_df.head(3))

## 2 — Report Table (mean ± std)

In [ ]:
from evaluation import summarise_results, build_report_table

summary      = summarise_results(results_df)
report_table = build_report_table(summary)

# Save
report_table.to_csv(RESULTS_DIR / 'report_table.csv', index=False)
print("Report table saved → results/report_table.csv")
display(report_table)

## 3 — AUC Heatmap

In [ ]:
# Pivot for heatmap: rows = method, cols = (strategy × label_fraction)
heatmap_data = (
    summary[["method", "strategy", "label_fraction", "test_auc"]]
    .assign(label_fraction=summary[('test_auc', 'label_fraction')])
    # Pivot manually
)

# Use the mean column
auc_mean = summary[('test_auc', 'mean')].values.reshape(4, 9)  # 4 methods × 9 combos
methods = summary[('method', '')].unique()[:4]
labels  = [f"{s} / {l}" for s, l in zip(
    summary[('strategy', '')],
    summary[('test_auc', 'label_fraction')]
)]

import numpy as np
methods_arr = summary.groupby('method').first().index.tolist()[:4]
strategies  = ['frozen', 'partial', 'full']
label_list  = ['1%', '5%', '10%']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, label in zip(axes, label_list):
    mask = summary[('test_auc', 'label_fraction')] == label
    sub  = summary[mask].set_index('method').loc[methods_arr]
    matrix = sub[[('test_auc', 'mean')]].values.reshape(3, 4).T  # strategy × method
    sns.heatmap(
        matrix,
        annot=True, fmt='.3f',
        xticklabels=strategies,
        yticklabels=methods_arr,
        ax=ax,
        vmin=0.65, vmax=0.95,
        cmap='YlOrRd'
    )
    ax.set_title(f'Test AUC — {label} labelled')
    ax.set_xlabel('Fine-tune strategy')
    ax.set_ylabel('Method')

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'auc_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4 — Per-Method Learning Curves (best strategy per method)

In [ ]:
# Plot val AUC progression across label fractions for each method
# We use the first seed's history from run_single_experiment
# If you re-ran with multiple seeds, extract history from the saved dicts.

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()

for ax, method in zip(axes, ['supervised_from_scratch', 'simclr', 'mae', 'mae_improved']):
    sub = results_df[results_df['method'] == method]
    for label in ['1%', '5%', '10%']:
        rows = sub[sub['label_fraction'] == label]
        means = rows.groupby('seed')['best_val_auc'].mean()
        # Show spread across seeds via a bar
        aucs = rows.groupby('seed')['test_auc'].mean()
        ax.bar(label, aucs.mean(), yerr=aucs.std(), capsize=5,
               label=label, alpha=0.8)
    ax.set_title(method)
    ax.set_xlabel('Label fraction')
    ax.set_ylabel('Test AUC')
    ax.set_ylim(0.5, 1.0)
    ax.legend(title='Label frac')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Test AUC by Method × Label Fraction', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'auc_by_method_label.png', dpi=150, bbox_inches='tight')
plt.show()

## 5 — ROC & Precision-Recall Curves (best model per method)

In [ ]:
# Load the full y_true / y_prob arrays from the best seed run.
# The raw arrays are stored in the saved result dicts.
# Here we re-compute them from the final model of the best config.

from data import load_all_data, build_downstream_loaders, LABEL_FRACTIONS
from training import build_simclr_classifier, build_supervised_scratch_classifier
from training import build_mae_classifier, build_mae_improved_classifier
from training import train_classifier, DEFAULT_EXPERIMENT_CONFIG, set_seed
from evaluation import evaluate_model

DRIVE_DATA_DIR = Path('/content/drive/MyDrive/ST456 Group project/pcamv1')
DATA_DIR        = Path('/content/pcam_data')

print('Loading data for best-model evaluation...')
data = load_all_data(DRIVE_DATA_DIR, DATA_DIR,
                     pretrain_fraction=0.15,
                     downstream_pool_fraction=0.15,
                     use_subset=True)
train_loaders, val_loader, test_loader = build_downstream_loaders(data)

best_configs = [
    ('supervised_from_scratch', 'full',   CHECKPOINT_DIR / 'mae_improved_encoder.pth'),
    ('simclr',                  'frozen', CHECKPOINT_DIR / 'simclr_encoder_tailored.pth'),
    ('mae',                     'frozen', CHECKPOINT_DIR / 'mae_encoder.pth'),
    ('mae_improved',            'frozen', CHECKPOINT_DIR / 'mae_improved_encoder.pth'),
]

set_seed(42)
roc_results = {}

for method, strategy, ckpt_path in best_configs:
    print(f'\nEvaluating {method} / {strategy}...')
    if method == 'supervised_from_scratch':
        model = build_supervised_scratch_classifier()
    elif method == 'simclr':
        model = build_simclr_classifier(ckpt_path)
    elif method == 'mae':
        model = build_mae_classifier(ckpt_path)
    else:
        model = build_mae_improved_classifier(ckpt_path)

    model, _, _, _ = train_classifier(
        model, train_loaders['10%'], val_loader, strategy=strategy,
        config=DEFAULT_EXPERIMENT_CONFIG,
    )
    metrics = evaluate_model(model, test_loader)
    roc_results[method] = {
        'y_true': metrics['y_true'],
        'y_prob': metrics['y_prob'],
        'test_auc': metrics['auc'],
    }
    print(f'  test_auc = {metrics["auc"]:.4f}')

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, auc

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['gray', 'blue', 'red', 'green']

for (method, res), color in zip(roc_results.items(), colors):
    fpr, tpr, _ = roc_curve(res['y_true'], res['y_prob'])
    precision, recall, _ = precision_recall_curve(res['y_true'], res['y_prob'])
    axes[0].plot(fpr, tpr, color=color, lw=2,
                  label=f"{method} (AUC={res['test_auc']:.3f})")
    axes[1].plot(recall, precision, color=color, lw=2,
                  label=f"{method} (AUC={res['test_auc']:.3f})")

axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves (10% labelled, full fine-tune)')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].grid(alpha=0.3)

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves (10% labelled)')
axes[1].legend(loc='upper right', fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 6 — t-SNE Visualisation

In [ ]:
from evaluation import extract_embeddings

# Use the same models already trained above (from roc_results)
# We reuse model objects from the loop (need to re-build them)
set_seed(42)

tsne_figure, axes = plt.subplots(2, 2, figsize=(12, 12))
axes = axes.flatten()

for ax, (method, strategy, ckpt_path) in zip(axes, best_configs):
    if method == 'supervised_from_scratch':
        model = build_supervised_scratch_classifier()
    elif method == 'simclr':
        model = build_simclr_classifier(ckpt_path)
    elif method == 'mae':
        model = build_mae_classifier(ckpt_path)
    else:
        model = build_mae_improved_classifier(ckpt_path)

    model, _, _, _ = train_classifier(
        model, train_loaders['10%'], val_loader, strategy=strategy,
        config=DEFAULT_EXPERIMENT_CONFIG,
    )

    embeddings, labels = extract_embeddings(model, test_loader, method)
    from evaluation import plot_tsne
    plot_tsne(embeddings, labels, title=f'{method} t-SNE')
    plt.savefig(FIGURE_DIR / f'tsne_{method}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f't-SNE for {method} saved.')

## 7 — Grad-CAM Interpretability (MAE Improved vs Supervised)

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from evaluation import GradCAM
from data import make_classification_transform

set_seed(42)

# Reload the two models we want to compare
model_sup = build_supervised_scratch_classifier()
model_sup, _, _, _ = train_classifier(
    model_sup, train_loaders['10%'], val_loader,
    strategy='full', config=DEFAULT_EXPERIMENT_CONFIG,
)

model_mae = build_mae_improved_classifier(
    CHECKPOINT_DIR / 'mae_improved_encoder.pth'
)
model_mae, _, _, _ = train_classifier(
    model_mae, train_loaders['10%'], val_loader,
    strategy='partial', config=DEFAULT_EXPERIMENT_CONFIG,
)

transform = make_classification_transform(data.channel_mean, data.channel_std)

# Pick 4 test images
indices = [0, 10, 50, 100]
fig, axes = plt.subplots(4, 5, figsize=(18, 14))

for row, idx in enumerate(indices):
    img = data.x_test[idx]          # HWC float [0,1]
    label = int(data.y_test[idx])
    img_tensor = transform(img).unsqueeze(0).to('cuda' if torch.cuda.is_available() else 'cpu')

    # Row: original, supervised-GC, mae-GC, supervised-GC overlay, mae-GC overlay
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f'True label: {label}')
    axes[row, 0].axis('off')

    # Supervised Grad-CAM
    cam_sup = GradCAM(model_sup.cuda(), model_sup.encoder.layer4)
    heat_sup = cam_sup(img_tensor.cuda(), class_idx=label)

    axes[row, 1].imshow(img)
    axes[row, 1].imshow(heat_sup, cmap='jet', alpha=0.4)
    axes[row, 1].set_title('Supervised (full) GC')
    axes[row, 1].axis('off')

    # MAE Improved Grad-CAM — Grad-CAM works on conv layers; MAE encoder is ViT-based.
    # We attach to the last decoder conv / patch embed instead.
    cam_mae = GradCAM(model_mae.cuda(), model_mae.encoder.patch_embed)
    heat_mae = cam_mae(img_tensor.cuda(), class_idx=label)

    axes[row, 2].imshow(img)
    axes[row, 2].imshow(heat_mae, cmap='jet', alpha=0.4)
    axes[row, 2].set_title('MAE Improved GC')
    axes[row, 2].axis('off')

plt.suptitle('Grad-CAM Comparison: Supervised vs MAE Improved')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'gradcam_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8 — Save All Figures & Final Summary

In [ ]:
import os

print('=' * 55)
print('  ANALYSIS COMPLETE — Saved outputs')
print('=' * 55)
print(f'\nResults:')
for f in sorted(os.listdir(RESULTS_DIR)):
    print(f'  {RESULTS_DIR}/{f}')

print(f'\nFigures:')
for f in sorted(os.listdir(FIGURE_DIR)):
    size = os.path.getsize(FIGURE_DIR / f) / 1e3
    print(f'  {FIGURE_DIR}/{f}  ({size:.0f} KB)')

print('\nAll done. These figures and tables can be used directly in the report.')